In [1]:
# 1. Install udocker and required tools
!pip install -q udocker
!udocker install

# 2. Verify udocker installation
!udocker version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 6.8 MB/s eta 0:00:00
Error: do not run as root !
Error: do not run as root !


In [2]:
# 1. Install Nginx and pyngrok
!apt-get update -qq --fix-missing
!apt-get install -y nginx -qq
!pip install -q pyngrok

# 2. Download and install ngrok CLI
!curl -sSL https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -o ngrok.tgz
!tar -xvzf ngrok.tgz -C /usr/local/bin
!rm -f ngrok.tgz
!ngrok --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package nginx-common.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../0-nginx-common_1.18.0-6ubuntu14.20_all.deb ...
Unpacking nginx-common (1.18.0-6ubuntu14.20) ...
Selecting previously unselected package libnginx-mod-http-geoip2.
Preparing to unpack .../1-libnginx-mod-http-geoip2_1.18.0-6ubuntu14.20_amd64.deb ...
Unpacking libnginx-mod-http-geoip2 (1.18.0-6ubuntu14.20) ...
Selecting previously unselected package libnginx-mod-http-image-filter.
Preparing to unpack .../2-libnginx-mod-http-image-filter_1.18.0-6ubuntu14.20_amd64.deb ...
Unpacking libnginx-mod-http-image-filter (1.18.0-6ubuntu14.20) ...
Selecting previously unselected package libnginx-mod-http-xslt-filter.
Preparing to unpack ...

In [3]:
from google.colab import userdata
from pyngrok import ngrok

# Retrieve and set the ngrok auth token from Colab Secrets
ngrok_token = userdata.get('YOUR_NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_token)

In [4]:
import subprocess

images = [
    "sreedockerhub19/helidon-demo:latest",
    "sreedockerhub19/helidon-demo-kotlin:latest",
    "sreedockerhub19/micronaut-java:latest",
    "sreedockerhub19/micronaut-kotlin:latest",
    "sreedockerhub19/simple-http-server:latest",
    "sreedockerhub19/simple-http-server-kotlin:latest",
    "sreedockerhub19/my-angular-app:latest",
    "sreedockerhub19/my-angular-app-main:latest",
    "sreedockerhub19/my-nextjs-app:latest",
    "sreedockerhub19/my-nextjs-app-ts-js:latest",
    "sreedockerhub19/my-nextjs-app-typescript:latest",
    "sreedockerhub19/ruby-http-server:latest",
    "sreedockerhub19/my-vertx-app:latest",
    "sreedockerhub19/my-spring-app:latest",
    "sreedockerhub19/my-spring-boot-app:latest",
    "sreedockerhub19/my-spring-boot-maven-app:latest",
    "sreedockerhub19/my-vertx-app-kotlin:latest",
    "sreedockerhub19/my-ktor-app:latest",
    "sreedockerhub19/my-javalin-app:latest",
    "sreedockerhub19/javalin-java-app:latest",
    "sreedockerhub19/my-quarkus-kotlin-app:latest",
    "sreedockerhub19/my-quarkus-java-app:latest"
]

print(f"Pulling and setting up {len(images)} images via udocker...\n")

for img in images:
    subprocess.run(["udocker", "--allow-root", "pull", img], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    container_name = img.split('/')[-1].replace(':', '_')
    subprocess.run(["udocker", "--allow-root", "create", "--name=" + container_name, img], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

!udocker --allow-root ps

Pulling and setting up 22 images via udocker...

CONTAINER ID                         P M NAMES              IMAGE               
028870e0-da54-3ceb-a913-5c549e6d966b . W ['helidon-demo_latest'] sreedockerhub19/helidon-demo:latest
9e3c6d07-45db-3e8c-b085-ec6d7818e42b . W ['my-spring-boot-app_latest'] sreedockerhub19/my-spring-boot-app:latest
e548824d-3166-3239-a5f2-e20f25313d98 . W ['my-angular-app_latest'] sreedockerhub19/my-angular-app:latest
63581bf9-8144-351b-bf07-0ba2777b45b6 . W ['micronaut-java_latest'] sreedockerhub19/micronaut-java:latest
4a6f553a-7600-3e85-b5d8-64013cb30efc . W ['my-quarkus-java-app_latest'] sreedockerhub19/my-quarkus-java-app:latest
f1b8e086-5c23-3cc5-a158-819f8a748a4e . W ['ruby-http-server_latest'] sreedockerhub19/ruby-http-server:latest
a497db21-3369-31c7-9902-7b852c5f781e . W ['my-nextjs-app-ts-js_latest'] sreedockerhub19/my-nextjs-app-ts-js:latest
dfe56fe3-8f74-3d32-9837-d0bf4c8b3818 . W ['my-ktor-app_latest'] sreedockerhub19/my-ktor-app:latest
afd077ce

In [17]:
from pyngrok import ngrok
import subprocess
import time
import socket

app_port_mapping = {
    # "my-quarkus-kotlin_latest": 8093,
    # "my-quarkus-java_latest": 8094,
    "ruby-http-server_latest": 10000,
    "my-angular-app_latest": 4200,
    "my-angular-app-main_latest": 4201,
    "my-nextjs-app_latest": 3000,
    "my-nextjs-app-ts-js_latest": 3001,
    "my-nextjs-app-typescript_latest": 3002,
    "my-vertx-app_latest": 8087,
    "my-spring-app_latest": 8088,
    "my-spring-boot-app_latest": 8089,
    "my-spring-boot-maven-app_latest": 8090,
    "my-vertx-app-kotlin_latest": 8091,
    "my-ktor-app_latest": 8092,
    "my-javalin-app_latest": 7000,
    "javalin-java-app_latest": 7001,
    "simple-http-server_latest": 8085,
    "helidon-demo_latest": 8081,
    "helidon-demo-kotlin_latest": 8082,
    "micronaut-java_latest": 8083,
    "micronaut-kotlin_latest": 8084,
    "simple-http-server-kotlin_latest": 8086,
}

apps_list = list(app_port_mapping.items())
batch_size = 3
batches = [apps_list[i:i + batch_size] for i in range(0, len(apps_list), batch_size)]

previous_processes = []

def wait_for_port(port, timeout=15):
    """Polls the port until the server actively accepts connections"""
    start_time = time.time()
    while time.time() - start_time < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1)
            if s.connect_ex(('localhost', port)) == 0:
                return True
        time.sleep(1)
    return False

for batch_index, current_batch in enumerate(batches, 1):
    print(f"\n==================================================")
    print(f"🔄 Processing Batch {batch_index} of {len(batches)}")
    print(f"==================================================")

    # Cleanup previous batch containers to avoid port collisions
    if previous_processes:
        print("🛑 Cleaning up previous batch containers...")
        for proc in previous_processes:
            proc.terminate()
            try:
                proc.wait(timeout=3)
            except subprocess.TimeoutExpired:
                proc.kill()
        time.sleep(3)

    ngrok.kill()
    active_tunnels = {}
    current_processes = []

    for container_name, host_port in current_batch:
        internal_port = (
            80 if "angular" in container_name else
            (3000 if "nextjs" in container_name else
            (10000 if "ruby" in container_name else 8080)))

        print(f"\n📦 Starting {container_name} -> Host Port {host_port}")

        # Smart execution logic based on application type
        if "simple-http-server" in container_name:
            # Special handling for classpath-based java apps
            proc = subprocess.Popen(
                [
                    "udocker", "--allow-root", "run",
                    f"--publish={host_port}:{internal_port}",
                    "--entrypoint=/bin/sh",
                    container_name,
                    "-c",
                    "java -jar app.jar || java -cp app.jar com.example.SimpleHttpServer"
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
        elif "quarkus" in container_name:
            # Quarkus standard runner jar: run from /app directly with host binding forced
            proc = subprocess.Popen(
                [
                    "udocker", "--allow-root", "run",
                    f"--publish={host_port}:{internal_port}",
                    "--entrypoint=/bin/sh",
                    container_name,
                    "-c",
                    "java -Dquarkus.http.host=0.0.0.0 -jar my-quarkus-java-app-1.0.0-runner.jar || java -Dquarkus.http.host=0.0.0.0 -jar quarkus-run.jar"
                ]
            )
        elif "ruby" in container_name:
            # Force Ruby to bind to 0.0.0.0 and use internal_port via command-line without touching server_.rb
            proc = subprocess.Popen(
                [
                    "udocker", "--allow-root", "run",
                    f"--publish={host_port}:{internal_port}",
                    "-e", f"PORT={internal_port}",
                    "--entrypoint=/bin/sh",
                    container_name,
                    "-c",
                    f"ruby -e \"ENV['PORT']='{internal_port}'; load 'server_.rb'\""
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
        elif any(jvm in container_name for jvm in ["helidon", "micronaut", "spring", "vertx", "ktor", "javalin"]):
            # JVM / Kotlin / Buildpack apps: force 0.0.0.0 binding via shell wrapper
            proc = subprocess.Popen(
                [
                    "udocker", "--allow-root", "run",
                    f"--publish={host_port}:{internal_port}",
                    "--entrypoint=/bin/sh",
                    container_name,
                    "-c",
                    "java -Dhelidon.webserver.host=0.0.0.0 -Dmicronaut.server.host=0.0.0.0 -jar app.jar"
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
        else:
            # Standard frontend / non-JVM apps (Angular, Next.js, Ruby, etc.)
            proc = subprocess.Popen(
                [
                    "udocker", "--allow-root", "run",
                    f"--publish={host_port}:{internal_port}",
                    container_name
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

        current_processes.append(proc)

        # Tailored wait times for heavier JVM / Quarkus / Javalin workloads
        if any(fw in container_name for fw in ["javalin", "quarkus", "kotlin"]):
            boot_wait_time = 20
            port_check_timeout = 30
        elif "java" in container_name or "spring" in container_name or "helidon" in container_name or "micronaut" in container_name:
            boot_wait_time = 20
            port_check_timeout = 25
        else:
            boot_wait_time = 15
            port_check_timeout = 15

        print(f"⏳ Waiting {boot_wait_time}s for {container_name} to finish booting...")
        time.sleep(boot_wait_time)

        # Double check with port polling
        if wait_for_port(host_port, timeout=port_check_timeout):
            print(f"✅ Port {host_port} confirmed open!")
        else:
            print(f"⚠️ Port {host_port} did not respond to ping within {port_check_timeout}s, connecting ngrok anyway...")

        try:
            public_url = ngrok.connect(host_port)
            active_tunnels[container_name] = public_url
            print(f"✨ Public URL -> {public_url}")
        except Exception as e:
            print(f"❌ Failed to create ngrok tunnel for {container_name}: {e}")

    print(f"\n--------------------------------------------------")
    print(f"✅ Batch {batch_index} active endpoints:")
    for name, url in active_tunnels.items():
        print(f"• {name} : {url}")

    previous_processes = current_processes

    if batch_index < len(batches):
        print(f"\n⏳ Waiting 15 seconds before moving to the next batch...")
        time.sleep(15)

print("\n🎉 All application batches processed successfully!")



🔄 Processing Batch 1 of 7

📦 Starting ruby-http-server_latest -> Host Port 10000
⏳ Waiting 15s for ruby-http-server_latest to finish booting...
✅ Port 10000 confirmed open!
✨ Public URL -> NgrokTunnel: "https://4afc-34-75-24-74.ngrok-free.app" -> "http://localhost:10000"

📦 Starting my-angular-app_latest -> Host Port 4200
⏳ Waiting 15s for my-angular-app_latest to finish booting...
✅ Port 4200 confirmed open!
✨ Public URL -> NgrokTunnel: "https://915e-34-75-24-74.ngrok-free.app" -> "http://localhost:4200"

📦 Starting my-angular-app-main_latest -> Host Port 4201
⏳ Waiting 15s for my-angular-app-main_latest to finish booting...
✅ Port 4201 confirmed open!
✨ Public URL -> NgrokTunnel: "https://c4aa-34-75-24-74.ngrok-free.app" -> "http://localhost:4201"

--------------------------------------------------
✅ Batch 1 active endpoints:
• ruby-http-server_latest : NgrokTunnel: "https://4afc-34-75-24-74.ngrok-free.app" -> "http://localhost:10000"
• my-angular-app_latest : NgrokTunnel: "https://

In [7]:
#!udocker --allow-root run helidon-demo-kotlin_latest